# Descarga de datos de movilidad (Montevideo / STM) a Volumes de Unity Catalog
Fuentes:
1. Viajes realizados en STM (CKAN, 1 recurso por mes)
2. Horarios de ómnibus por parada (CKAN)
3. Horarios de ómnibus por puntos de control (CKAN)
4. Localización de paradas (GeoNetwork, uuid c6ea0476-9804-424a-9fae-2ac8ce2eee31)

Requisito: el cluster/serverless necesita salida a internet hacia ckan.montevideo.gub.uy,
ckan-data.montevideo.gub.uy, datos-abiertos.montevideo.gub.uy, imnube.montevideo.gub.uy y geoweb.montevideo.gub.uy.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "movilidad")
dbutils.widgets.text("volume", "raw")
dbutils.widgets.text("desde_anio_mes", "")        # ej: 2024-01  (vacío = todo)
dbutils.widgets.text("paradas_url", "")           # URL directa de la capa de paradas, si ya la tenés
dbutils.widgets.dropdown("descomprimir", "true", ["true", "false"])

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
DESDE = dbutils.widgets.get("desde_anio_mes").strip()
PARADAS_URL = dbutils.widgets.get("paradas_url").strip()
DESCOMPRIMIR = dbutils.widgets.get("descomprimir") == "true"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print("Destino:", BASE)

In [ ]:
import os, re, zipfile, requests, shutil, json
from urllib.parse import unquote, urlparse

CKAN = "https://ckan.montevideo.gub.uy"
S = requests.Session()
S.headers["User-Agent"] = "Mozilla/5.0 (databricks-ingest)"
manifest = []   # (dataset, recurso, url, destino, estado)


def ckan_resources(slug):
    r = S.get(f"{CKAN}/api/3/action/package_show", params={"id": slug}, timeout=60)
    r.raise_for_status()
    return r.json()["result"]["resources"]


def nombre_archivo(resp, url, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*?=(?:UTF-8'')?\"?([^\";]+)", cd)
    if m:
        return unquote(m.group(1))
    base = os.path.basename(urlparse(url).path)
    return unquote(base) if "." in base else fallback


def descargar(url, carpeta, fallback, dataset, recurso):
    os.makedirs(carpeta, exist_ok=True)
    try:
        with S.get(url, stream=True, timeout=300, allow_redirects=True) as r:
            r.raise_for_status()
            ctype = r.headers.get("content-type", "")
            if "text/html" in ctype and not url.endswith((".txt", ".html")):
                manifest.append((dataset, recurso, url, "", "HTML_NO_ARCHIVO (link de share, bajar a mano o probar ?download)"))
                return None
            destino = os.path.join(carpeta, nombre_archivo(r, url, fallback))
            if os.path.exists(destino) and os.path.getsize(destino) > 0:
                manifest.append((dataset, recurso, url, destino, "YA_EXISTIA"))
                return destino
            tmp = destino + ".part"
            with open(tmp, "wb") as f:
                shutil.copyfileobj(r.raw, f, length=1024 * 1024)
            os.replace(tmp, destino)
            manifest.append((dataset, recurso, url, destino, "OK"))
            return destino
    except Exception as e:
        manifest.append((dataset, recurso, url, "", f"ERROR: {e}"))
        return None


def descomprimir(path):
    if DESCOMPRIMIR and path and path.lower().endswith(".zip"):
        try:
            with zipfile.ZipFile(path) as z:
                z.extractall(path[:-4])
        except Exception as e:
            print("No se pudo descomprimir", path, e)

## 1. Viajes realizados en STM (mensual)

In [ ]:
MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}


def anio_mes(nombre):
    m = re.search(r"(" + "|".join(MESES) + r")\s+(?:del?\s+)?(\d{4})", nombre.lower())
    return (int(m.group(2)), MESES[m.group(1)]) if m else None


slug = "viajes-realizados-en-los-omnibus-del-sistema-de-transporte-metropolitano-stm"
carpeta = f"{BASE}/viajes_stm"
for res in ckan_resources(slug):
    nombre = res["name"]
    am = anio_mes(nombre)
    if am and DESDE:
        y, m = map(int, DESDE.split("-"))
        if am < (y, m):
            continue
    fallback = f"viajes_stm_{am[0]}{am[1]:02d}.zip" if am else re.sub(r"\W+", "_", nombre) + ".bin"
    p = descargar(res["url"], carpeta, fallback, "viajes_stm", nombre)
    descomprimir(p)

In [ ]:
import os
carpeta = f"{BASE}/viajes_stm"
nombres = {
    "ztluldakqdwvxbfzeo3wsg": "viajes_stm_012023",
    "qhhhkadnq3kfwd4x67rypa": "viajes_stm_022023",
    "f_ke9soltjillcbpt-huwq": "viajes_stm_032023",
    "o7elqy0qsei9edql3le8gw": "viajes_stm_042023",
    "xi-_agzwt9afpgbhysquya": "viajes_stm_052023",
}
for viejo, nuevo in nombres.items():
    for ext in ["", ".zip"]:
        a = f"{carpeta}/{viejo}{ext}"
        if os.path.exists(a):
            os.rename(a, f"{carpeta}/{nuevo}{ext}")

## 2 y 3. Horarios (por parada y por puntos de control)
Estos se actualizan diariamente: conviene guardar en una subcarpeta con fecha.

In [ ]:
from datetime import date

hoy = date.today().isoformat()
for slug, sub in [("horarios-de-omnibus-urbanos-por-parada-stm", "horarios_por_parada"),
                  ("horarios-de-omnibus-urbanos-por-puntos-de-control-stm", "horarios_puntos_control")]:
    carpeta = f"{BASE}/{sub}/{hoy}"
    for res in ckan_resources(slug):
        fallback = re.sub(r"\W+", "_", res["name"]) + ".bin"
        p = descargar(res["url"], carpeta, fallback, sub, res["name"])
        descomprimir(p)

## 4. Localización de paradas (GeoNetwork)
Si no pasás `paradas_url`, se intenta descubrir los enlaces de descarga/WFS desde el registro de metadatos.

In [ ]:
UUID = "c6ea0476-9804-424a-9fae-2ac8ce2eee31"
GEO = "https://geoweb.montevideo.gub.uy/geonetwork/srv"
carpeta = f"{BASE}/paradas"

if not PARADAS_URL:
    candidatos = set()
    for ruta, acc in [(f"{GEO}/api/records/{UUID}", "application/xml"),
                      (f"{GEO}/api/records/{UUID}/formatters/xml", "application/xml")]:
        try:
            t = S.get(ruta, headers={"Accept": acc}, timeout=60).text
            for u in re.findall(r"https?://[^\s<>\"']+", t):
                if re.search(r"wfs|geoserver|\.zip|\.shp|\.csv|\.geojson|download|ows", u, re.I):
                    candidatos.add(u.rstrip(").,;"))
        except Exception as e:
            print("Fallo", ruta, e)
    print("Enlaces candidatos encontrados en la metadata:")
    for u in sorted(candidatos):
        print("  ", u)
    print("\nElegí el correcto y volvé a correr con el widget 'paradas_url'.")
else:
    p = descargar(PARADAS_URL, carpeta, "paradas_stm.bin", "paradas", "capa_paradas")
    descomprimir(p)

## Resumen

In [ ]:
import pandas as pd
df = pd.DataFrame(manifest, columns=["dataset", "recurso", "url", "destino", "estado"])
display(spark.createDataFrame(df))
df.to_csv(f"{BASE}/_manifest_descarga.csv", index=False)
print(df["estado"].str.split(":").str[0].value_counts())